## import mentions data

In [1]:
import pandas as pd
import json

#file = "../data/mentions/narrative_mentions.jsonl"
file = '/kaggle/input/datasets/lianestrauch/narrative-mentions-jsonl/narrative_mentions.jsonl'

records = []

with open(file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            print(f"Skipping malformed line in {file}")

mentions_df = pd.DataFrame(records)

print(mentions_df.shape)
print(mentions_df.columns)

(537645, 15)
Index(['paperId', 'doi', 'oa_id', 'text_type', 'text_cleaned', 'match_type',
       'matched_seq', 'matched_char_start', 'matched_char_end', 'sentence_pos',
       'pos', 'id', 'context_token_ids', 'context_n_tokens',
       'matched_token_indices'],
      dtype='object')


## change the tokenizer

In [2]:
from transformers import AutoTokenizer, AutoModel


MODEL_NAME = "allenai/scibert_scivocab_uncased"
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create the columns first
mentions_df["context_token_ids"] = None
mentions_df["context_n_tokens"] = None
mentions_df["matched_token_indices"] = None

for i, row in mentions_df.iterrows():
    context = row["text_cleaned"]

    encoding = TOKENIZER(
        context,
        add_special_tokens=True,
        return_offsets_mapping=True,
    )

    token_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]

    char_start = row["matched_char_start"]
    char_end = row["matched_char_end"]

    # Tokens whose character span overlaps the matched sequence
    # Special tokens ([CLS]/[SEP]) have offset (0, 0)
    matched_token_indices = [
        idx
        for idx, (start, end) in enumerate(offsets)
        if end > char_start
        and start < char_end
        and not (start == 0 and end == 0)
    ]

    mentions_df.at[i, "context_token_ids"] = token_ids
    mentions_df.at[i, "context_n_tokens"] = len(token_ids)
    mentions_df.at[i, "matched_token_indices"] = matched_token_indices

mentions_df.head()


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

,paperId,doi,oa_id,text_type,text_cleaned,match_type,matched_seq,matched_char_start,matched_char_end,sentence_pos,pos,id,context_token_ids,context_n_tokens,matched_token_indices
0,0000278bfcd5dfc1d586156ccff16e168f238e83,10.1093/CWW/VPAB030,https://openalex.org/W3196316976,title,reading contemporary black british and african...,full_match,narrative,85,94,-1,0,0,"[102, 5589, 12610, 3778, 9816, 137, 7608, 3258...",18,[15]
1,0000278bfcd5dfc1d586156ccff16e168f238e83,10.1093/CWW/VPAB030,https://openalex.org/W3196316976,abstract,"in their introduction, wyatt and george expres...",full_match,narrative,83,92,2,0,1,"[102, 121, 547, 2067, 422, 18829, 4576, 137, 1...",47,[17]
2,4aae5a329915daacf79c4354ec5c72d136424348,10.1162/AFAR_R_00470,https://openalex.org/W2945533467,abstract,the authors show how visual culture has been e...,full_match,narratives,175,185,33,0,2,"[102, 111, 1991, 405, 539, 2180, 2343, 434, 52...",71,[42]
3,4aaeb618b4e06b682c54f3db0eace3690b07c01a,10.1159/000532024,https://openalex.org/W4385651398,title,vocabulary diversity in personal narratives pr...,full_match,narratives,33,43,-1,0,3,"[102, 14401, 4715, 121, 4026, 25733, 2772, 121...",27,[5]
4,4aaeb618b4e06b682c54f3db0eace3690b07c01a,10.1159/000532024,https://openalex.org/W4385651398,abstract,introduction: this study examines whether ther...,full_match,narratives,148,158,0,0,4,"[102, 2067, 862, 238, 527, 15817, 1681, 461, 2...",34,[23]


## set up and testing GPUs

In [3]:
!nvidia-smi

Thu Aug 27 12:33:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch
from transformers import AutoTokenizer, AutoModel

# Check GPU count
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPUs available: {torch.cuda.device_count()}")  # Should print 2

tokenizer = TOKENIZER
model = AutoModel.from_pretrained(MODEL_NAME)

# Send model to main GPU first
model = model.to(device)

# Wrap model to use ALL available GPUs
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

model.eval()

# Process data
texts = ["First test sentence.", "Second test sentence."]
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Note: Access outputs directly as usual
print(outputs.last_hidden_state.shape)

GPUs available: 2


pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 7, 768])


## Update the get_embeddings function

In [5]:
#  another update to optimize the code for GPU usage
import numpy as np

def get_token_embeddings_batched(sentences, target_indices_list, tokenizer, model, device):
    """
    Extracts concatenated/selected layer embeddings for specific token indices per sentence in a batch.
    
    sentences: list of str, e.g., ["The cat sat.", "A fast dog ran."]
    target_indices_list: list of lists/tuples, e.g., [[2], [1, 2]] matching sentence sequence
    """
    # 1. Tokenize as a batch with padding enabled
    # Truncation is fine as long as target_indices aren't past max_length (e.g. 512)
    inputs = tokenizer(
        sentences, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    ).to(device)

    # 2. Forward pass on GPU
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # hidden_states tuple length: 13 (for bert-base: embeddings + 12 layers)
    # Stack the last 4 layers -> shape: (4, batch_size, seq_len, 768)
    layers = torch.stack(outputs.hidden_states[-4:])

    batch_embeddings = []

    # 3. Extract requested indices for each sentence in the batch
    for batch_idx, indices in enumerate(target_indices_list):
        if len(indices) == 1:
            # Shape: (4, 768) -> (4th-to-last, ..., last layer) for token at indices[0]
            emb = layers[:, batch_idx, indices[0]]
        else:
            # Shape: (4, num_tokens_selected, 768)
            emb = layers[:, batch_idx, indices[0]: indices[-1] + 1]
        
        # Move back to CPU memory and convert to Float32 NumPy array
        batch_embeddings.append(emb.to(torch.float32).cpu().numpy())

    # --- CRITICAL: Delete tensors and clear CUDA cache ---
    del inputs, outputs, layers
    torch.cuda.empty_cache()

    return batch_embeddings

## create a wrapper for the get_embeddings function

In [6]:
# garbage collection to avoid overflowing GPU
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"Allocated memory: {torch.cuda.memory_allocated() / 1e6:.2f} MB")
# Should ideally print ~0.00 MB

Allocated memory: 450.79 MB


In [7]:
from pathlib import Path
import time
from datetime import datetime

embeddings_path = Path("bert-base")
embeddings_path.mkdir(parents=True, exist_ok=True)
progress_path = embeddings_path / "progress.json"

# TODO: LOAD mentions_df!!!


if progress_path.is_file():
    with open(progress_path, "r") as f:
        progress = json.load(f)

    # check to what extent the data has been processed - create to_process_df
    to_process_df = mentions_df[mentions_df["id"].isin(progress["to_process"])] # TODO: make this more robust! (e.g. what if the key does not exist?)
    print(f"1: {len(to_process_df)}")
    # run a quick sum check
    print(f"All papers match: {len(mentions_df) == progress['all_ids']}")
    print(f"Papers to process match: {len(to_process_df) == progress['to_process_n']}")
    ids_done_prior = progress['ids_processed_n'] # for the chunks_naming

    # PROCESS THE DATA

else:
    progress = {}
    progress["all_ids"] = len(mentions_df)
    progress["ids_processed_n"] = 0
    ids_done_prior = progress['ids_processed_n'] #for the chunks naming
    
    # Filter - context >512
    large_context_ids = list(mentions_df[mentions_df.context_n_tokens > 512]["id"])
    progress["ids_rm_len_n"] = len(large_context_ids)
    progress["ids_rm_len"] = large_context_ids
    to_process_df = mentions_df.drop(mentions_df[mentions_df.context_n_tokens > 512].index)
    print(f"2: {len(to_process_df)}")

    # Filter - keep only "narrative" and "narratives"
    non_conservative_match = to_process_df[~to_process_df["matched_seq"].isin(["narrative", "narratives"])]["id"].tolist()
    progress["ids_rm_match_n"] = len(non_conservative_match)
    progress["ids_rm_match"] = non_conservative_match
    to_process_df = to_process_df.drop(to_process_df[to_process_df.id.isin(non_conservative_match)].index)
    print(f"3: {len(to_process_df)}")


    # store the to_process_df info
    progress["to_process_n"] = len(to_process_df)
    progress["to_process"] = to_process_df.id.to_list()

    # ----> START HERE WITH all rows that are not been 

    with open(progress_path, "w") as f:
        json.dump(progress, f, indent=4)

##################################################################
# Work with to_process_df
# index into the (4, ...) stack: 0=4th-to-last, 1=3rd-to-last, 2=2nd-to-last, 3=last
LAYER_COLS = {
    "layer_last": 3,
    "layer_2nd_last": 2,
    "layer_3rd_last": 1,
    "layer_4th_last": 0,
}
to_process_df = to_process_df.reset_index(drop=True)
Path(embeddings_path / "chunks").mkdir(parents=True, exist_ok=True)

# set up time logging
progress["start_time"] = datetime.now().isoformat()
start_ts = time.monotonic()  # for computing elapsed duration cheaply

BATCH_SIZE = 32  # Power of 2 for T4 GPUs

# Prepare records array
records = []
total_rows = len(to_process_df)

# Convert DataFrame columns to lists for fast slicing
all_texts = to_process_df["text_cleaned"].tolist()
all_indices = to_process_df["matched_token_indices"].tolist()
all_ids = to_process_df["id"].tolist()

# Process in batch strides
for start_idx in range(0, total_rows, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_rows)
    
    # Slice the current batch
    batch_texts = all_texts[start_idx:end_idx]
    batch_indices = all_indices[start_idx:end_idx]
    batch_ids = all_ids[start_idx:end_idx]

    # Get embeddings for the batch on GPU
    # Returns a list of NumPy arrays (one array per sentence in batch)
    batch_embeddings = get_token_embeddings_batched(
        sentences=batch_texts,
        target_indices_list=batch_indices,
        tokenizer=tokenizer,
        model=model,
        device=device
    )

    # Process results from this batch
    for row_id, embeddings in zip(batch_ids, batch_embeddings):
        record = {"id": row_id}
        
        for col_name, layer_idx in LAYER_COLS.items():
            layer_emb = embeddings[layer_idx]
            
            # Fix for non-conservative matches (multi-token spans):
            # layer_emb shape is (num_tokens, 768) if multi-token, or (768,) if single token.
            if layer_emb.ndim > 1:
                layer_emb = layer_emb.mean(axis=0)  # Average token span to 1D vector
                
            record[col_name] = layer_emb.tolist()
            
        records.append(record)

    # Logging progress
    processed_count = len(records)
    if (start_idx + BATCH_SIZE) % 1000 < BATCH_SIZE:
        print(f"Processed: {min(start_idx + BATCH_SIZE, total_rows):,} / {total_rows:,}")

    # Checkpoint every 4,000 items
    if len(records) >= 4000:
        chunk_df = pd.DataFrame(records)
        processed_ids = {r["id"] for r in records}
        records = []  # Reset buffer

        # Save parquet chunk
        chunk_file = embeddings_path / "chunks" / f"chunk_{start_idx + BATCH_SIZE + ids_done_prior}.parquet"
        chunk_df.to_parquet(chunk_file, index=False)

        print(f"--- Saved Checkpoint: {chunk_file.name} ---")

        # Fast update of progress file using set operations
        progress["ids_processed_n"] += len(processed_ids)
        progress["to_process"] = list(set(progress["to_process"]) - processed_ids)
        progress["to_process_n"] = len(progress["to_process"])
        
        # Update elapsed time
        progress["end_time"] = datetime.now().isoformat()
        progress["elapsed_seconds"] = progress.get("elapsed_seconds", 0) + round(time.monotonic() - start_ts, 1)
        start_ts = time.monotonic()  # Reset step timer

        # Persist progress to disk
        with open(progress_path, "w") as f:
            json.dump(progress, f, indent=4)
            
        # empty cache before continuing
        gc.collect()
        torch.cuda.empty_cache()

# Handle leftover rows that didn't reach a full 5k chunk
if records:
    chunk_df = pd.DataFrame(records)
    processed_ids = {r["id"] for r in records}
    chunk_df.to_parquet(embeddings_path / "chunks" / "chunk_final.parquet", index=False)
    
    # Final progress update
    progress["ids_processed_n"] += len(processed_ids)
    progress["to_process"] = list(set(progress["to_process"]) - processed_ids)
    progress["to_process_n"] = len(progress["to_process"])
    progress["end_time"] = datetime.now().isoformat()
    
    with open(progress_path, "w") as f:
        json.dump(progress, f, indent=4)
        
    print(f"Finished! Processed all {total_rows:,} items.")


2: 537340
3: 499195
Processed: 1,024 / 499,195
Processed: 2,016 / 499,195
Processed: 3,008 / 499,195
Processed: 4,000 / 499,195
--- Saved Checkpoint: chunk_4000.parquet ---
Processed: 5,024 / 499,195
Processed: 6,016 / 499,195
Processed: 7,008 / 499,195
Processed: 8,000 / 499,195
--- Saved Checkpoint: chunk_8000.parquet ---
Processed: 9,024 / 499,195
Processed: 10,016 / 499,195
Processed: 11,008 / 499,195
Processed: 12,000 / 499,195
--- Saved Checkpoint: chunk_12000.parquet ---
Processed: 13,024 / 499,195
Processed: 14,016 / 499,195
Processed: 15,008 / 499,195
Processed: 16,000 / 499,195
--- Saved Checkpoint: chunk_16000.parquet ---
Processed: 17,024 / 499,195
Processed: 18,016 / 499,195
Processed: 19,008 / 499,195
Processed: 20,000 / 499,195
--- Saved Checkpoint: chunk_20000.parquet ---
Processed: 21,024 / 499,195
Processed: 22,016 / 499,195
Processed: 23,008 / 499,195
Processed: 24,000 / 499,195
--- Saved Checkpoint: chunk_24000.parquet ---
Processed: 25,024 / 499,195
Processed: 26,0

In [8]:
import json
from pathlib import Path

progress_path = embeddings_path / "progress.json"


with open(progress_path, "r") as f:
        progress = json.load(f)

progress.keys()

dict_keys(['all_ids', 'ids_processed_n', 'ids_rm_len_n', 'ids_rm_len', 'ids_rm_match_n', 'ids_rm_match', 'to_process_n', 'to_process', 'start_time', 'end_time', 'elapsed_seconds'])

In [9]:
print(progress["ids_rm_match_n"])
print(progress["to_process_n"])
print(progress["elapsed_seconds"]/60)

38145
0
54.021666666666675


In [10]:
total_to_process = progress["all_ids"] - progress["ids_rm_len_n"] - progress["ids_rm_match_n"]
print(f"total to process (before even procesing anything): {total_to_process}")

print(f"already processed: {progress["ids_processed_n"]}")

print(f"still to be processed should be: {total_to_process - progress["ids_processed_n"]}")
print(f"according to the process file, the number of mentions still to process are: {progress["to_process_n"]}")
print(f"Which is confusing, since the length of all the ids still to process stored in the json is: {len(progress["to_process"])}")

print("debug fixed - nothing is confusing anymore")


total to process (before even procesing anything): 499195
already processed: 499195
still to be processed should be: 0
according to the process file, the number of mentions still to process are: 0
Which is confusing, since the length of all the ids still to process stored in the json is: 0
debug fixed - nothing is confusing anymore
